In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings("ignore")

base_dir = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))

# CSV paths
diagnoses_path = os.path.join(base_dir, "data", "diagnoses.csv")
patients_path  = os.path.join(base_dir, "data", "patients.csv")
outcomes_path  = os.path.join(base_dir, "data", "outcomes.csv")

from src.data_pipeline import load_and_merge_data, clean_and_impute
from src.features import engineer_features

# Loading Data
df = load_and_merge_data(diagnoses_path, patients_path, outcomes_path)
admitted = clean_and_impute(df)
X, y, feature_cols = engineer_features(admitted)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Loading Artifacts
stacking_model = joblib.load("../models/best_model.pkl")
best_threshold = joblib.load("../models/best_threshold.pkl")

# Extracting base XGBoost model from the Stacking Classifier for exact Tree SHAP
xgb_model = stacking_model.named_estimators_["xgb"]

stacking_probs = stacking_model.predict_proba(X_test)[:, 1]
stacking_preds = (stacking_probs >= best_threshold).astype(int)

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test)
sv           = shap_values.values

print(f"SHAP values shape: {sv.shape}")
print(f"Features: {X_test.shape[1]}  |  Test samples: {X_test.shape[0]}")

In [ ]:
mean_abs_shap = np.abs(sv).mean(axis=0)
importance_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": mean_abs_shap,
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nTop 20 features by mean |SHAP|:")
print(importance_df.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 7))
top20 = importance_df.head(20)
bars  = ax.barh(
    top20["feature"][::-1],
    top20["importance"][::-1],
    color="#378ADD", edgecolor="none",
)
ax.set_xlabel("Mean |SHAP value| — average impact on readmission probability")
ax.set_title("Top 20 features driving 30-day readmission (XGBoost SHAP)")
ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20, show=False)
plt.title("SHAP beeswarm — feature impact direction and magnitude")
plt.tight_layout()
plt.show()

shap.summary_plot(sv, X_test, feature_names=feature_cols, max_display=20, show=False)
plt.title("SHAP summary — readmission risk drivers")
plt.tight_layout()
plt.show()

In [ ]:
top4 = importance_df["feature"].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flat, top4):
    feat_idx = feature_cols.index(feat)
    shap.dependence_plot(
        feat_idx, sv, X_test,
        feature_names=feature_cols,
        ax=ax, show=False,
    )
    ax.set_title(f"SHAP dependence — {feat}")

plt.suptitle("How top features affect readmission risk", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
high_risk_idx   = np.argmax(stacking_probs)

print(f"\nHighest-risk patient index: {high_risk_idx}")
print(f"  Stacking predicted prob : {stacking_probs[high_risk_idx]:.3f}")
print(f"  Actual label            : {int(y_test.iloc[high_risk_idx])}")

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[high_risk_idx], max_display=15, show=False)
plt.title(f"Patient #{high_risk_idx} — high-risk waterfall (predicted prob: {stacking_probs[high_risk_idx]:.2f})")
plt.tight_layout()
plt.show()

low_risk_idx = np.argmin(stacking_probs)

print(f"\nLowest-risk patient index: {low_risk_idx}")
print(f"  Stacking predicted prob : {stacking_probs[low_risk_idx]:.3f}")
print(f"  Actual label            : {int(y_test.iloc[low_risk_idx])}")

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[low_risk_idx], max_display=15, show=False)
plt.title(f"Patient #{low_risk_idx} — low-risk waterfall (predicted prob: {stacking_probs[low_risk_idx]:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(7, 6))
frac_pos, mean_pred = calibration_curve(y_test, stacking_probs, n_bins=10)
ax.plot(mean_pred, frac_pos, marker="o", lw=1.5, color="#7F77DD", label="Stacking (RF+XGB)", markersize=4)

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual rate)")
ax.set_title("Calibration curve — are predicted probabilities trustworthy?")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Error Analysis
y_test_arr = y_test.values

false_negatives = X_test[(stacking_preds == 0) & (y_test_arr == 1)]
false_positives = X_test[(stacking_preds == 1) & (y_test_arr == 0)]
true_positives  = X_test[(stacking_preds == 1) & (y_test_arr == 1)]

print("\n── Error analysis ─────────────────────────────────────────────────────")
print(f"False negatives (missed readmissions) : {len(false_negatives)}")
print(f"False positives (false alarms)        : {len(false_positives)}")
print(f"True positives  (correct flags)       : {len(true_positives)}")

compare_cols = ["age", "charlson_index", "length_of_stay_days", "num_comorbidities", "icu_admission"]

summary = pd.DataFrame({
    "False negatives": false_negatives[compare_cols].mean(),
    "False positives": false_positives[compare_cols].mean(),
    "True positives":  true_positives[compare_cols].mean(),
}).round(2)

print("\nAverage profile of each error type:")
print(summary.to_string())